# TASK 4 - Prediction

## Libraries and Dependencies
This section imports all necessary libraries for our machine learning pipeline:
- Basic data manipulation: pandas, numpy
- Machine learning: scikit-learn (for models and metrics)
- Deep learning: PyTorch
- Sampling methods: imbalanced-learn
- Visualization: matplotlib

In [1]:
import os
import json
import pickle
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from imblearn.under_sampling import RandomUnderSampler, TomekLinks, EditedNearestNeighbours, ClusterCentroids
from imblearn.over_sampling import ADASYN, SMOTE
from tqdm import tqdm

from sklearn.svm import SVC

import matplotlib.pyplot as plt
from IPython.display import clear_output

from sklearn.ensemble import BaggingClassifier, AdaBoostClassifier

## File Path Configuration
Define paths for:
- Model logging and serialization
- Input dataset locations (cleaned data, train/test splits)
These paths will be used throughout the notebook for data loading and model saving.

In [2]:
file_name = "logs/best_models.json"
model_path = 'logs/'
dataset = '../dataset/cleaned.csv'
train_path = '../dataset/train.csv'
test_path = '../dataset/test.csv'

## Data Preprocessing and Label Creation
This section:
1. Loads the cleaned cycling race dataset
2. Creates binary classification labels (1 for top 20 positions, 0 otherwise)
3. Checks class distribution to assess imbalance

In [ ]:
df_cleaned = pd.read_csv(dataset, skipinitialspace=True)
df_for_prediction = df_cleaned.copy()

df_for_prediction['label'] = df_for_prediction.apply(lambda row: 1 if row['position'] <= 20 else 0, axis=1)
df_for_prediction.head()

print(len(df_for_prediction[df_for_prediction['label'] == 1]))
print(len(df_for_prediction[df_for_prediction['label'] == 0]))

t = len(df_for_prediction[df_for_prediction["profile"] == 1])

## Feature Selection
Remove features that:
- Are direct indicators of performance (position, points)
- Contain unuseful information
- Are identifiers (IDs, names)
- Have potential data leakage
This ensures our model learns from relevant predictive features only.

In [4]:
features_to_remove = [
    'position',  
    'points', 
    'position_range', 
    'cyclist_points_range',

    'delta', 
    'average_temperature',
    'uci_points',
    'cyclist_team',

    'name_race',
    'name_race_norm',
    'name_cyclist',
    'id_race', 
    'id_cyclist',
    'cyclist',

    'time',
    'nationality'
]
for feature in features_to_remove:
    if feature in df_for_prediction.columns:
        df_for_prediction.drop(columns=[feature], inplace=True)

In [ ]:
df_for_prediction.info()
print(len(df_for_prediction[df_for_prediction['label'] == 1]))
print(len(df_for_prediction[df_for_prediction['label'] == 0]))

In [6]:
def oneHotFeaturesEncoding():
    features_to_encode = [
    'cyclist_age_range',
    'startlist_quality_range',
    'profile'
    ]
    # Use pd.get_dummies() to one-hot encode the categorical columns
    df_pandas_encoded = pd.get_dummies(df_cleaned, columns=features_to_encode, drop_first=True)
    print(f"One-Hot Encoded Data using Pandas:\n{df_pandas_encoded}\n")
    return df_pandas_encoded

## Train-Test Split
Split data based on time:
- Training data: Races before 2022
- Test data: Races from 2022 onwards

In [7]:
df_for_prediction['date_for_splitting'] = pd.to_datetime(df_for_prediction['date'])
train = df_for_prediction[df_for_prediction['date_for_splitting'].dt.year < 2022]
test = df_for_prediction[df_for_prediction['date_for_splitting'].dt.year >= 2022]

fields_to_remove = ['date', 'date_for_splitting']
for field in fields_to_remove:
    if field in train.columns:
        train = train.drop(columns=[field])
    if field in test.columns:
        test = test.drop(columns=[field])

In [ ]:
print(len(train))
print(len(test))

train.to_csv(train_path, index=False)
test.to_csv(test_path, index=False)

In [ ]:
X_train, y_train = train.drop(columns=['label']), train[['label']]
X_test, y_test = test.drop(columns=['label']), test[['label']]

X_train, X_eval, y_train, y_eval = train_test_split(X_train, y_train, test_size=0.5, random_state=42)
train.head()

## Sampling Strategy Implementation
Define different sampling approaches to handle class imbalance:
- ADASYN: Adaptive Synthetic Sampling
- SMOTE: Synthetic Minority Over-sampling Technique
- Combined approaches using undersampling and oversampling
Each strategy aims to create a more balanced dataset for training.

In [12]:
class Sampling:
    ADASYN = 'ADASYN'
    SMOTE = 'SMOTE'
    UNDER_RANDOM_OVER_SMOTE = 'UNDER_RANDOM_OVER_SMOTE'
    UNDER_TOMEK_OVER_SMOTE = 'UNDER_TOMEK_OVER_SMOTE'
    UNDER_ENN_OVER_SMOTE = 'UNDER_ENN_OVER_SMOTE'
    UNDER_CL_OVER_SMOTE = 'UNDER_CL_OVER_SMOTE'


def sampling(X, y, samp):
    print(samp)
    if samp == 'ADASYN':
        adasyn = ADASYN(sampling_strategy='minority', random_state=42)
        X_resampled, y_resampled = adasyn.fit_resample(X, y)
        X = X_resampled
        y = y_resampled

    elif samp == 'SMOTE':
        smote = SMOTE(sampling_strategy='minority', random_state=42)
        X_resampled, y_resampled = smote.fit_resample(X, y)
        X = X_resampled
        y = y_resampled

    elif samp == 'UNDER_RANDOM_OVER_SMOTE':
        undersample = RandomUnderSampler(sampling_strategy=0.5, random_state=42)
        X_resampled, y_resampled = undersample.fit_resample(X, y)
        smote = SMOTE(sampling_strategy=1.0, random_state=42)
        X_balanced, y_balanced = smote.fit_resample(X_resampled, y_resampled)
        X = X_balanced
        y = y_balanced
        
    elif samp == 'UNDER_TOMEK_OVER_SMOTE':
        tl = TomekLinks()
        X_tl, y_tl = tl.fit_resample(X, y)
        undersample = RandomUnderSampler(sampling_strategy=0.5, random_state=42)
        X_resampled, y_resampled = undersample.fit_resample(X_tl, y_tl)
        smote = SMOTE(sampling_strategy=1.0, random_state=42)
        X_balanced, y_balanced = smote.fit_resample(X_resampled, y_resampled)
        X = X_balanced
        y = y_balanced

    elif samp == 'UNDER_ENN_OVER_SMOTE':
        enn = EditedNearestNeighbours(n_neighbors=3)
        X_enn, y_enn = enn.fit_resample(X, y)
        undersample = RandomUnderSampler(sampling_strategy=0.5, random_state=42)
        X_resampled, y_resampled = undersample.fit_resample(X_enn, y_enn)
        smote = SMOTE(sampling_strategy=1.0, random_state=42)
        X_balanced, y_balanced = smote.fit_resample(X_resampled, y_resampled)
        X = X_balanced
        y = y_balanced

    elif samp == 'UNDER_CL_OVER_SMOTE':
        cc = ClusterCentroids(random_state=42)
        X_cc, y_cc = cc.fit_resample(X, y)
        undersample = RandomUnderSampler(sampling_strategy=0.5, random_state=42)
        X_resampled, y_resampled = undersample.fit_resample(X_cc, y_cc)
        smote = SMOTE(sampling_strategy=1.0, random_state=42)
        X_balanced, y_balanced = smote.fit_resample(X_resampled, y_resampled)
        X = X_balanced
        y = y_balanced
    else:
        raise Exception("oversampling type does not allow")
    print('label == 1: ', len(y[y['label'] == 1]))
    print('label == 0: ', len(y[y['label'] == 0]))
    return X, y

## Model Performance Tracking
Implement functions to:
1. Track and log model results including:
   - Model type
   - Hyperparameters
   - Performance metrics
   - Classification reports
2. Save trained models for future use

In [15]:
def track_results(model_name: str, loss: float, accuracy: float, report: str, hyperparams, dataset: str):
    file = open(file_name, 'r')
    objs: list = json.load(file)
    file.close()

    file = open(file_name, 'w')
    try:
        obj = {
            "model": model_name,
            "hyperparams": hyperparams,
            "loss": loss,
            "accuracy": accuracy,
            "report": [line for line in report.split('\n')],
            "dataset": dataset
        }
        print(obj)
        objs.append(obj)
        print(objs)
        json.dump(objs, file)
        file.close()
    except:
        json.dump(objs, file)
        file.close()
        raise Exception("Data not saved in the logs file")

In [16]:
def save_model_as_pickle(model, model_name):
    with open(f'{model_path}/{model_name}.pkl', 'wb') as file:
        pickle.dump(model, file)


## Model Selection and Hyperparameter Tuning
For each model type (Decision Tree, KNN, SVM, Neural Network, Bagging, Boosting):
1. Define hyper-parameter grids for search
2. Implement cross-validation
3. Test different sampling strategies
4. test the model on validation and test set
5. Track best performing configurations
This systematic approach helps identify optimal model settings.

### Decision Tree 

In [ ]:
param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [None, 3, 5, 10],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

sample_types = [
    Sampling.UNDER_TOMEK_OVER_SMOTE,
    Sampling.UNDER_ENN_OVER_SMOTE,
    Sampling.UNDER_RANDOM_OVER_SMOTE,
    Sampling.SMOTE,
    Sampling.ADASYN
]
best_accuracy_for_dataset_type = 0
best_hyperparams_for_dataset_type = {}
best_model_for_dataset_type = None
best_dataset_type = ''
best_accuracy_for_type = 0
best_results = dict()

for sample_type in sample_types:
    X_train_sampled, y_train_sampled = sampling(X_train, y_train, sample_type)
    clf = DecisionTreeClassifier(random_state=42)
    grid_search = GridSearchCV(estimator=clf, param_grid=param_grid, scoring='accuracy', cv=2, verbose=1, return_train_score=True)
    grid_search.fit(X_train_sampled, y_train_sampled)
    print(grid_search.best_score_)
    if best_accuracy_for_type < grid_search.best_score_:
        best_hyperparams_for_dataset_type = grid_search.best_params_
        best_model_for_dataset_type = grid_search.best_estimator_
        best_accuracy_for_type = grid_search.best_score_
        best_dataset_type = sample_type
        best_results = grid_search.cv_results_

In [ ]:
train_loss = 1 - np.array(best_results['mean_train_score'])
eval_loss = 1 - np.array(best_results['mean_test_score'])

# Plot the training and evaluation loss
plt.figure(figsize=(10, 6))
plt.plot(train_loss, label='Training Loss', marker='o')
plt.plot(eval_loss, label='Evaluation Loss', marker='x')
plt.title('Training and Evaluation Loss during Grid Search')
plt.xlabel('Hyperparameter Configuration Index')
plt.ylabel('Loss')
plt.legend()
plt.grid()
plt.show()

In [ ]:
# Evaluate the best model on the eval set
best_model = DecisionTreeClassifier(random_state=42, **best_hyperparams_for_dataset_type)
print("Best Hyperparameters:", best_hyperparams_for_dataset_type)
X_train_sampled, y_train_sampled = sampling(X_train, y_train, best_dataset_type)
best_model.fit(X_train_sampled, y_train_sampled)
# Evaluate on the eval set
y_eval_pred = best_model.predict(X_eval)
eval_accuracy = accuracy_score(y_eval, y_eval_pred)
print("\nEvaluation Accuracy:", eval_accuracy)
print("Classification Report on Eval Set:\n", classification_report(y_eval, y_eval_pred))

# Model assessment: Final evaluation on the test set
y_test_pred = best_model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_test_pred)
report = classification_report(y_test, y_test_pred)
print("\nTest Accuracy:", test_accuracy)
print("Classification Report on Test Set:\n", report)

In [ ]:
save_model_as_pickle(best_model, 'best_decision_tree')
track_results("Decision Tree", None, test_accuracy, report, best_hyperparams_for_dataset_type, best_dataset_type)

### Neightest Neighbor Classifier

In [ ]:
# Define the Model
knn = KNeighborsClassifier()

# Hyperparameter Tuning using GridSearchCV
param_grid = {'n_neighbors': np.arange(1, 31), 'weights': ['uniform', 'distance'], 'metric': ['euclidean', 'manhattan']}

sample_types = [
    Sampling.UNDER_TOMEK_OVER_SMOTE,
    Sampling.UNDER_ENN_OVER_SMOTE,
    Sampling.UNDER_RANDOM_OVER_SMOTE,
    # Sampling.SMOTE,
    # Sampling.ADASYN
]
best_accuracy_for_dataset_type = 0
best_hyperparams_for_dataset_type = {}
best_model_for_dataset_type = None
best_dataset_type = ''

for sample_type in sample_types:
    X_train_sampled, y_train_sampled = sampling(X_train, y_train, sample_type)
    knn = KNeighborsClassifier()
    grid_search = GridSearchCV(knn, param_grid, cv=5, scoring='accuracy', verbose=1)
    grid_search.fit(X_train_sampled, y_train_sampled['label'].to_numpy())
    if best_accuracy_for_dataset_type < grid_search.best_score_:
        best_hyperparams_for_dataset_type = grid_search.best_params_
        best_model_for_dataset_type = grid_search.best_estimator_
        best_accuracy_for_dataset_type = grid_search.best_score_
        best_dataset_type = sample_type
# Best Parameters and Best Score
print("Best Parameters:", best_hyperparams_for_dataset_type)
print("Best Cross-Validation Accuracy:", best_accuracy_for_dataset_type)


In [ ]:
# Evaluate the Model on the Test Set
best_knn = KNeighborsClassifier(**best_hyperparams_for_dataset_type)
X_train_sampled, y_train_sampled = sampling(X_train, y_train, best_dataset_type)
best_knn.fit(X_train_sampled, y_train_sampled['label'].to_numpy())
y_pred = best_knn.predict(X_test)

test_accuracy = accuracy_score(y_test, y_pred)
print("Test Set Accuracy:", test_accuracy)

# Detailed Classification Report and Confusion Matrix
report  = classification_report(y_test, y_pred)
print("Classification Report:\n", report)

print(best_hyperparams_for_dataset_type)
best_hyperparams_for_dataset_type['metric'] = str(best_hyperparams_for_dataset_type['metric'])
best_hyperparams_for_dataset_type['n_neighbors'] = int(best_hyperparams_for_dataset_type['n_neighbors'])
best_hyperparams_for_dataset_type['weights'] = str(best_hyperparams_for_dataset_type['weights'])
save_model_as_pickle(best_knn, 'best_knn')
track_results("knn", None, test_accuracy, report, best_hyperparams_for_dataset_type, best_dataset_type)

### SVM for classification

In [ ]:
# Hyperparameter Tuning using GridSearchCV
param_grid = {
    'C': [0.1, 1, 10, 100],
    'kernel': ['linear', 'poly', 'rbf', 'sigmoid'],
    'gamma': ['scale', 'auto']
}

sample_types = [
    Sampling.UNDER_TOMEK_OVER_SMOTE,
    Sampling.UNDER_ENN_OVER_SMOTE,
    Sampling.UNDER_RANDOM_OVER_SMOTE,
    Sampling.SMOTE,
    Sampling.ADASYN
]
best_accuracy_for_dataset_type = 0
best_hyperparams_for_dataset_type = {}
best_model_for_dataset_type = None
best_dataset_type = ''

for sample_type in sample_types:
    X_train_sampled, y_train_sampled = sampling(X_train, y_train, sample_type)
    svc = SVC()
    grid_search = GridSearchCV(svc, param_grid, cv=5, scoring='accuracy', verbose=1)
    grid_search.fit(X_train_sampled, y_train_sampled['label'].to_numpy())
    print(grid_search.best_score_)
    if best_accuracy_for_dataset_type < grid_search.best_score_:
        best_hyperparams_for_dataset_type = grid_search.best_params_
        best_model_for_dataset_type = grid_search.best_estimator_
        best_accuracy_for_dataset_type = grid_search.best_score_
        best_dataset_type = sample_type
# Best Parameters and Best Score
print("Best Parameters:", best_hyperparams_for_dataset_type)
print("Best Cross-Validation Accuracy:", best_accuracy_for_dataset_type)

In [ ]:
# Evaluate the Model on the Test Set
print(best_dataset_type)
best_svc = SVC(**best_hyperparams_for_dataset_type)
X_train_sampled, y_train_sampled = sampling(X_train, y_train, best_dataset_type)
best_svc.fit(X_train_sampled, y_train_sampled['label'].to_numpy())

y_pred = best_svc.predict(X_test)

test_accuracy = accuracy_score(y_test, y_pred)
print("Test Set Accuracy:", test_accuracy)

# Detailed Classification Report and Confusion Matrix
report = classification_report(y_test, y_pred)

print("Classification Report:\n", report)
print("Confusion Matrix:\n", confusion_matrix)

save_model_as_pickle(best_svc, 'best_svc')
track_results("svc", None, test_accuracy, report, best_hyperparams_for_dataset_type, best_dataset_type)

### Neural Network Classifier

In [21]:
def print_losses(train_loss, eval_loss):
    x = [i for i in range(len(train_loss))]

    plt.plot(x, train_loss, label='train loss', color='blue')
    plt.plot(x, eval_loss, label='val loss', color='red')
    plt.ylim(0, 1)
    # Add labels, legend, and grid
    plt.xlabel('epochs')
    plt.ylabel('errors')
    plt.title('Overlapping losses functions')
    plt.legend()
    plt.grid(True)

    # Show the plot
    plt.show()

In [22]:
'''
class NeuralNetwork(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super(NeuralNetwork, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, num_classes)
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x
'''

class NeuralNetwork(nn.Module):
    def __init__(self, input_dim, layer_sizes):
        super(NeuralNetwork, self).__init__()
        layers = []
        layers.append(nn.Linear(input_dim, layer_sizes[0]))
        layers.append(nn.ReLU())
        for i in range(len(layer_sizes) - 1):
            layers.append(nn.Linear(layer_sizes[i], layer_sizes[i+1]))
            if i < len(layer_sizes) - 2:
                layers.append(nn.ReLU())
        layers.append(nn.Sigmoid())
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

In [ ]:
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print(f"Using device: {device}")

In [24]:
def map_dtype_to_torch(dtype):
    """Map pandas dtypes to corresponding PyTorch dtypes."""
    if np.issubdtype(dtype, np.floating):
        return torch.float32
    elif np.issubdtype(dtype, np.integer):
        return torch.long
    elif np.issubdtype(dtype, np.int64):
        return torch.int64
    elif np.issubdtype(dtype, np.bool_):
        return torch.bool
    else:
        print(dtype)
        raise TypeError(f"Unsupported dtype {dtype} for conversion to PyTorch tensor.")

In [25]:
def dataframe_to_tensordataset(X_df, y_df):
    """
    Convert pandas DataFrame to PyTorch TensorDataset, automatically adapting dtypes.
    Handles multi-target issues by ensuring the target tensor is 1D.
    """
    X_tensors = []
    for col in X_df.columns:
        torch_dtype = map_dtype_to_torch(X_df[col].dtype)
        X_tensors.append(torch.tensor(X_df[col].values, dtype=torch_dtype))
    X_tensor = torch.stack(X_tensors, dim=1)
    if len(y_df.shape) > 1 and y_df.shape[1] == 1:
        y_df = y_df.squeeze()
    elif len(y_df.shape) > 1:
        raise ValueError("Multi-target tensors are not supported. Provide a single target column.")
    y_dtype = map_dtype_to_torch(y_df.dtypes)
    y_tensor = torch.tensor(y_df.values, dtype=y_dtype)
    return TensorDataset(X_tensor, y_tensor)

In [26]:
def print_report(report):
    for line in report.split('\n'):
        print(line)

def stop_criterinon(loss, steps):
    steps = [loss] + steps[:len(steps) - 1]
    check = [steps[i] > steps[i+1] for i in range(len(steps) - 1)]
    if check.count(True) == len(steps) - 1:
        return True, steps
    return False, steps

In [27]:
# Training loop
def train(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(loader)
    return avg_loss

# Evaluation loop
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    all_preds = []
    all_targets = []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            total_loss += loss.item()
            preds = outputs.argmax(dim=1)
            correct += (preds == y_batch).sum().item()
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(y_batch.cpu().numpy())
    report = classification_report(all_targets, all_preds, output_dict=False)
    return total_loss / len(loader), correct / len(loader.dataset), report

# Cross-validation for model selection
def cross_validate(dataset, input_dim, num_classes, k_folds=5):
    hidden_dims = [
        (32, num_classes),          # 1 hidden layer with 64 neurons
        (32, 32, num_classes),     # 2 hidden layers with 32 and 32 neurons
        (64, 32, num_classes),     # 2 hidden layers with 64 and 32 neurons
        (64, 64, num_classes),     # 2 hidden layers with 128 and 64 neurons
        (64, 32, 16, num_classes),  # 3 hidden layers with 64, 32, and 16 neurons
        (32, 32, 32, num_classes),  # 3 hidden layers with 32, 32, and 32 neurons
        (64, 64, 64, num_classes),  # 3 hidden layers with 64, 64, and 64 neurons
    ]
    learning_rates = [0.001, 0.01]
    weight_decays = [0.0001, 0.00001]
    epochs = 100
    steps = [388305.4695] * 10

    best_model = None
    best_accuracy = 0
    best_hyperparams = None

    kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)
    data = dataset[:]
    X_tensor, y_tensor = data[0], data[1]
    test = 0
    for hidden_dim in hidden_dims:
        for lr in learning_rates:
            for weight_decay in weight_decays:
                fold_accuracies = []
                test += 1
                print('test: ', test)
                for train_idx, val_idx in kf.split(X_tensor):
                    X_train_fold, y_train_fold = X_tensor[train_idx], y_tensor[train_idx]
                    X_val_fold, y_val_fold = X_tensor[val_idx], y_tensor[val_idx]

                    train_loader = DataLoader(TensorDataset(X_train_fold, y_train_fold), batch_size=32, shuffle=True)
                    val_loader = DataLoader(TensorDataset(X_val_fold, y_val_fold), batch_size=32)

                    model = NeuralNetwork(input_dim, hidden_dim).to(device)
                    criterion = nn.CrossEntropyLoss()
                    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

                    for epoch in tqdm(range(epochs)):  # Train for a few epochs per fold
                        train_loss = train(model, train_loader, criterion, optimizer, device)
                        check, steps = stop_criterinon(train_loss, steps)
                        if check:
                            break

                    _, val_accuracy, report = evaluate(model, val_loader, criterion, device)
                    fold_accuracies.append(val_accuracy)
                    print_report(report)

                avg_accuracy = np.mean(fold_accuracies)
                print(f"First hidden Dim={hidden_dim}, LR={lr}, Avg Accuracy={avg_accuracy:.4f}")

                if avg_accuracy > best_accuracy:
                    best_accuracy = avg_accuracy
                    best_hyperparams = (hidden_dim, lr, weight_decay)
                    best_model = NeuralNetwork(input_dim, hidden_dim).to(device)

    print(f"Best Model: First Hidden Dim={best_hyperparams[0]}, LR={best_hyperparams[1]}, Weight Decay={best_hyperparams[2]}, Accuracy={best_accuracy:.4f}")
    return best_model, best_hyperparams, best_accuracy

input_dim = X_train.shape[1]
num_classes = len(y_train['label'].unique())


In [ ]:
# Model selection (cross-validation)
sample_types = [
    Sampling.UNDER_TOMEK_OVER_SMOTE,
    Sampling.UNDER_ENN_OVER_SMOTE,
    Sampling.UNDER_RANDOM_OVER_SMOTE,
    # Sampling.SMOTE,
    # Sampling.ADASYN
]
best_accuracy_for_dataset_type = 0
best_hyperparams_for_dataset_type = {}
best_model_for_dataset_type = None
best_train_dataset = {}
best_dataset_type = ''

for sample_type in sample_types:
    X_train_sampled, y_train_sampled = sampling(X_train, y_train, sample_type)
    X_train_sampled = oneHotFeaturesEncoding(X_train_sampled)
    train_dataset = dataframe_to_tensordataset(X_train_sampled, y_train_sampled)
    best_model, best_hyperparams, best_accuracy = cross_validate(train_dataset, input_dim, num_classes)
    if best_accuracy_for_dataset_type < best_accuracy:
        best_hyperparams_for_dataset_type = best_hyperparams
        best_model_for_dataset_type = best_model
        best_accuracy_for_dataset_type = best_accuracy
        best_train_dataset = train_dataset
        best_dataset_type = sample_type

In [ ]:
best_dataset_type = Sampling.UNDER_ENN_OVER_SMOTE
X_train_sampled, y_train_sampled = sampling(X_train, y_train, best_dataset_type)
X_train_sampled = oneHotFeaturesEncoding(X_train_sampled)
train_dataset = dataframe_to_tensordataset(X_train_sampled, y_train_sampled)
eval_dataset = dataframe_to_tensordataset(X_eval, y_eval)
test_dataset = dataframe_to_tensordataset(X_test, y_test)

In [ ]:
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
best_hyperparams = best_hyperparams_for_dataset_type
best_model = NeuralNetwork(input_dim, best_hyperparams[0]).to(device)

print(f"Using device: {device}")
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(best_model.parameters(), lr=best_hyperparams[1], weight_decay=best_hyperparams[2])

epochs = 200
steps = [388305.4695] * 10
train_losses = []
eval_losses = []
for epoch in range(epochs):
    train_loss = train(best_model, train_loader, criterion, optimizer, device)
    eval_loader = DataLoader(eval_dataset, batch_size=32)
    eval_loss, eval_accuracy, eval_report = evaluate(best_model, eval_loader, criterion, device)
    clear_output(wait=True)
    print(f"Epoch {epoch+1}/{epochs}: Train Loss={train_loss:.4f}, Eval Loss={eval_loss:.4f}, Eval Accuracy={eval_accuracy:.4f}")
    print_report(eval_report)
    check, steps = stop_criterinon(eval_loss, steps)
    if check:
        break
    train_losses.append(train_loss)
    eval_losses.append(eval_loss)
    print_losses(train_losses, eval_losses)
# Test the model
test_loader = DataLoader(test_dataset, batch_size=32)
test_loss, test_accuracy, test_report = evaluate(best_model, test_loader, criterion, device)
print_report(test_report)
print(f"Test Loss={test_loss:.4f}, Test Accuracy={test_accuracy:.4f}")

In [ ]:
best_hyperparams_dict = {
    "hidden_layers": best_hyperparams[0],
    "learning_rate": best_hyperparams[1],
    "weight_decay": best_hyperparams[2],
}

nn_path = os.path.join(model_path, f"best_neural_network.pth")
torch.save(best_model.state_dict(), nn_path)
print(f"Model with layer configuration {best_hyperparams_dict} saved to {nn_path}.")
track_results("Neural Network", test_loss, test_accuracy, test_report, best_hyperparams_dict, best_dataset_type)

### Ensamble of models: Bagging

In [ ]:
bagging_params = {
    'n_estimators': [10, 50, 100],
    'max_samples': [0.5, 0.7, 1.0],
}
sample_types = [
    Sampling.UNDER_TOMEK_OVER_SMOTE,
    Sampling.UNDER_ENN_OVER_SMOTE,
    Sampling.UNDER_RANDOM_OVER_SMOTE,
    Sampling.SMOTE,
    Sampling.ADASYN
]
best_accuracy_for_dataset_type = 0
best_hyperparams_for_dataset_type = {}
best_model_for_dataset_type = None
best_dataset_type = ''

for sample_type in sample_types:
    X_train_sampled, y_train_sampled = sampling(X_train, y_train, sample_type)
    bagging = BaggingClassifier(random_state=42)
    grid_bagging = GridSearchCV(bagging, bagging_params, cv=5, scoring='accuracy')
    grid_bagging.fit(X_train_sampled, y_train_sampled['label'].to_numpy())
    if best_accuracy_for_dataset_type < grid_bagging.best_score_:
        best_hyperparams_for_dataset_type = grid_bagging.best_params_
        best_model_for_dataset_type = grid_bagging.best_estimator_
        best_accuracy_for_dataset_type = grid_bagging.best_score_
        best_dataset_type = sample_type

print(f"Best parameters for Bagging: {best_hyperparams_for_dataset_type}")

bagging_best = best_model_for_dataset_type
y_pred_bagging = bagging_best.predict(X_test)

print("Bagging Model Assessment:")
print(classification_report(y_test, y_pred_bagging))

In [ ]:
X_train_sampled, y_train_sampled = sampling(X_train, y_train, best_dataset_type)
bagging_best = BaggingClassifier(**best_hyperparams_for_dataset_type)
bagging_best.fit(X_train_sampled, y_train_sampled['label'].to_numpy())

print(f"Best parameters for Bagging: {best_hyperparams_for_dataset_type}")

y_pred_bagging = bagging_best.predict(X_test)

print("Bagging Model Assessment:")
test_accuracy = accuracy_score(y_test, y_pred_bagging)
report = classification_report(y_test, y_pred_bagging)

save_model_as_pickle(bagging_best, 'best_bagging')
print_report(report)
track_results("Bagging", None, test_accuracy, report, best_hyperparams_for_dataset_type, best_dataset_type)

### Ensamble of models: Boosting

In [ ]:
boosting_params = {
    'n_estimators': [10, 50, 100, 200, 500],
    'learning_rate': [0.0001, 0.001, 0.01, 0.1, 1.0],
    'algorithm': ['SAMME']
}
sample_types = [
    Sampling.UNDER_TOMEK_OVER_SMOTE,
    Sampling.UNDER_ENN_OVER_SMOTE,
    Sampling.UNDER_RANDOM_OVER_SMOTE,
    Sampling.SMOTE,
    Sampling.ADASYN
]

best_accuracy_for_dataset_type = 0
best_hyperparams_for_dataset_type = {}
best_model_for_dataset_type = None
best_dataset_type = ''

for sample_type in sample_types:
    X_train_sampled, y_train_sampled = sampling(X_train, y_train, sample_type)
    boosting = AdaBoostClassifier()
    grid_boosting = GridSearchCV(boosting, boosting_params, n_jobs=-1, cv=5, scoring='accuracy')
    grid_boosting.fit(X_train_sampled, y_train_sampled['label'].to_numpy())
    
    if best_accuracy_for_dataset_type < grid_boosting.best_score_:
        best_hyperparams_for_dataset_type = grid_boosting.best_params_
        best_model_for_dataset_type = grid_boosting.best_estimator_
        best_accuracy_for_dataset_type = grid_boosting.best_score_
        best_dataset_type = sample_type

print(f"Best parameters for Boosting: {best_hyperparams_for_dataset_type}")

boosting_best = best_hyperparams_for_dataset_type
y_pred_boosting = best_model_for_dataset_type.predict(X_test)

print("Boosting Model Assessment:")
print(classification_report(y_test, y_pred_boosting))
print(best_dataset_type)

In [ ]:
X_train_sampled, y_train_sampled = sampling(X_train, y_train, best_dataset_type)
boosting = AdaBoostClassifier(**best_hyperparams_for_dataset_type)
boosting.fit(X_train_sampled, y_train_sampled['label'].to_numpy())
y_pred = boosting.predict(X_test)

print("Bagging Model Assessment:")
test_accuracy = accuracy_score(y_test, y_pred)
report  = classification_report(y_test, y_pred)

print_report(report)
save_model_as_pickle(boosting, 'best_boosting')
track_results("Boosting", None, test_accuracy, report, best_hyperparams_for_dataset_type, best_dataset_type)